# Chapter 00 — Setup & System Architecture

**Patent RAG Engineering Masterclass.** This series builds, from first principles to a traced
production pipeline, a **patent-domain Retrieval-Augmented Generation system**: parsing real
patent XML/PDF, normalizing text while preserving provenance, structural chunking, sparse +
dense + hybrid retrieval, reranking, evaluation, grounded generation, an agent, guardrails, and
end-to-end observability — all over a **real, bundled corpus of US patents**.

Each chapter is **independently executable on a fresh Google Colab VM**. Cross-chapter state
travels through `artifacts/`, rebuilt on demand from `data/` by `patentrag.bootstrap.ensure()`,
so no chapter depends on another's kernel.

This chapter: the architecture, what each subsystem owns, the environment report, the
**Library Decision Table** (with the versions actually installed), and a tour of the corpus.

In [1]:
# === Chapter 00 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 00 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 00 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 1. What we are building

Two views of the same system. First, the **base retrieval → generation pipeline**:

```
Patent sources (XML / PDF)
      │  parsing / OCR                         ← Ch 02, 03
      ▼
Canonical document representation              ← Ch 01  (PatentDocument, …)
      │  Unicode normalization + OffsetMap      ← Ch 04  (provenance preserved)
      ▼
Section / claim structural parsing
      │  chunking + contextual enrichment       ← Ch 05
      ▼
 ┌──────────────────┬───────────────────┐
 │ Inverted index   │ Vector index      │
 │ BM25  (Ch 06)    │ Dense / HNSW (Ch 07)│
 └────────┬─────────┴─────────┬─────────┘
          │  hybrid retrieval  │
          ▼                    ▼
              RRF fusion  (Ch 08)
                   │
          Cross-encoder reranking (Ch 08)
                   │
            Context selection      (Ch 10)
                   │
                  LLM              (Ch 10)
                   ▼
        Grounded answer + citations
```

Then the **guardrailed agentic system** that wraps it (Ch 11–13):

```
USER
 │  INPUT GUARDRAILS      (PII, injection, safety, scope)     ← Ch 12
 ▼
AGENT / QUERY PLANNER      (tools + trajectory)               ← Ch 11
 │
RETRIEVAL
 │  RETRIEVAL GUARDRAILS   (data≠instruction, authorization)  ← Ch 12
 ▼
TOOLS
 │  TOOL GUARDRAILS        (schema, params, permissions)      ← Ch 12
 ▼
LLM
 │  OUTPUT GUARDRAILS      (groundedness, citations, PII, safety)
 ▼
ANSWER + TRACE + CITATIONS  (observability)                   ← Ch 13
```

### What each subsystem owns

| Subsystem | Owns | Chapter |
|---|---|---|
| Parsing / OCR | Turning XML & PDF bytes into text + structure + coordinates | 02, 03 |
| Document model | The canonical, structured `PatentDocument` (claims, sections, biblio) | 01 |
| Normalization | Search-optimized *shadow text* + a reversible `OffsetMap` to the original | 04 |
| Chunking | Retrieval/citation units that respect structure and carry anchors | 05 |
| Sparse retrieval | Exact-term matching (BM25) — technical terms, identifiers, claim language | 06 |
| Dense retrieval | Semantic matching (embeddings), ANN indexing, quantization | 07 |
| Fusion / rerank | Combining rankings (RRF) and precision reranking (cross-encoder) | 08 |
| Evaluation | Measuring retriever, RAG, and agent quality with real metrics | 09–11 |
| Generation | Context construction + citation-aware grounded answers | 10 |
| Agent | Tool selection, structured tool calls, trajectory capture | 11 |
| Guardrails | Five independent trust boundaries (defense in depth) | 12 |
| Observability | Per-request tracing, latency budget, provenance resolution | 13 |

## 2. Environment report

Everything below runs **CPU-only** and requires **no paid API key**. A GPU, if present, is used
opportunistically and never required; embedding/reranker inference is pinned to CPU for
determinism.

In [2]:
import pandas as pd
env = bs.environment_report()
pd.DataFrame([env]).T.rename(columns={0: "value"})

,value
python,3.12.10
platform,Windows-11-10.0.26200-SP0
in_colab,False
cpu_count,24
torch,2.12.0.dev20260304+cu130
cuda_available,True
tesseract,True


## 3. Library Decision Table

Selections were made against **current** PyPI / official docs (researched 2026-08) and
**smoke-tested before pinning** — not chosen from memory. Rationale lives in `DECISIONS.md`;
the machine-readable pins in `requirements.txt`. Below we print the versions **actually
installed in this kernel**, so the table is self-verifying.

In [3]:
import importlib.metadata as m
rows = [
    ("XML parse / XPath / validation", "lxml", "xml.etree, xmlschema", "Full XPath + DTD/XSD, C-speed"),
    ("PDF text / coords", "PyMuPDF", "pypdf, pdfminer, pypdfium2", "Fast words+bbox+blocks"),
    ("OCR", "pytesseract", "easyocr, rapidocr, PaddleOCR", "Maintained, per-word confidence, light"),
    ("Embeddings", "sentence-transformers", "bge-small, e5-small, gte-small", "MiniLM: reproducible CPU baseline"),
    ("Sparse / BM25", "rank-bm25", "bm25s, Pyserini/Lucene", "Pure-Python, zero-native (teaching)"),
    ("ANN / HNSW / PQ", "faiss-cpu", "hnswlib, ScaNN, Annoy", "HNSW + quantization in one dep"),
    ("PII", "presidio-analyzer", "regex, GLiNER, HF NER", "Recognizers + reversible anonymization"),
    ("Language ID", "langdetect", "fasttext, lingua", "Pure-Python, deterministic w/ seed"),
]
def ver(pkg):
    try: return m.version(pkg)
    except Exception: return "—"
df = pd.DataFrame(rows, columns=["Problem", "Selected", "Alternatives considered", "Why selected"])
df["Installed version"] = df["Selected"].map(ver)
df

,Problem,Selected,Alternatives considered,Why selected,Installed version
0,XML parse / XPath / validation,lxml,"xml.etree, xmlschema","Full XPath + DTD/XSD, C-speed",6.1.1
1,PDF text / coords,PyMuPDF,"pypdf, pdfminer, pypdfium2",Fast words+bbox+blocks,1.27.2.3
2,OCR,pytesseract,"easyocr, rapidocr, PaddleOCR","Maintained, per-word confidence, light",0.3.13
3,Embeddings,sentence-transformers,"bge-small, e5-small, gte-small",MiniLM: reproducible CPU baseline,6.0.0
4,Sparse / BM25,rank-bm25,"bm25s, Pyserini/Lucene","Pure-Python, zero-native (teaching)",0.2.2
5,ANN / HNSW / PQ,faiss-cpu,"hnswlib, ScaNN, Annoy",HNSW + quantization in one dep,1.15.0
6,PII,presidio-analyzer,"regex, GLiNER, HF NER",Recognizers + reversible anonymization,2.2.364
7,Language ID,langdetect,"fasttext, lingua","Pure-Python, deterministic w/ seed",1.0.9


The reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`), prompt-injection and content-safety
detectors, and the RAG/agent-eval approaches are introduced in their own chapters (08, 12, 09–11)
with the same "what's current / what we chose / why" treatment.

## 4. Corpus tour

The bundled corpus is **real, public US patents** deliberately on-theme with the technologies
this series teaches (retrieval, ANN, embeddings, indexing, NLP), fetched from Google Patents and
committed under `data/corpus/` so Colab never re-fetches. Full provenance is in
`data/PROVENANCE.md`. `ensure("corpus_raw")` validates the bundle is present.

In [4]:
manifest = bs.ensure("corpus_raw")
docs = bs.ensure("docs_canonical")
print(f"corpus: {manifest['n_docs']} patents · {manifest['n_claims']} claims · {manifest['n_sections']} sections")

rows = []
for d in docs:
    rows.append({
        "publication": d.publication_number,
        "title": d.title[:52],
        "CPC": d.cpc[0] if d.cpc else "",
        "claims": len(d.claims),
        "indep": len(d.independent_claims),
        "pub_date": str(d.publication_date),
        "inventors": len(d.inventors),
    })
corpus_df = pd.DataFrame(rows).sort_values("publication").reset_index(drop=True)
corpus_df

corpus: 15 patents · 304 claims · 109 sections


,publication,title,CPC,claims,indep,pub_date,inventors
0,US10083169B1,Topic-based sequence modeling neural networks,G06F17/276,15,3,2018-09-25,6
1,US10261954B2,Optimizing search result snippet selection,G06F,19,3,2019-04-16,3
2,US11093561B2,Fast indexing with graphs and compact regressi...,G06F,19,3,2021-08-17,3
3,US11113744B2,Personalized item recommendations through larg...,G06Q,18,2,2021-09-07,6
4,US11216459B2,Multi-layer semantic search,G06F,20,3,2022-01-04,2
5,US11257279B2,Systems and methods for providing non-parametr...,G06T,21,3,2022-02-22,1
6,US11714853B2,Efficient storage and searching of vector data...,G06F,25,4,2023-08-01,4
7,US11971885B2,Retrieval aware embedding,G06N,18,3,2024-04-30,8
8,US12099542B2,Implementing a graphical user interface to col...,G06F,13,4,2024-09-24,5
9,US8930304B2,Knowledge discovery from citation networks,G06N,17,3,2015-01-06,2


Note the genuine detail this real corpus carries — full independent/dependent claim structure,
real CPC codes, real dates, and real inventor names (including Unicode names like *Mert Öz* /
*Herwig Häle*, which we will use in the Unicode-normalization chapter). This is exactly the
structure a flat-text pipeline would throw away.

In [5]:
# One document up close: structure that "just text" would destroy.
d = next(x for x in docs if x.doc_id == "US11971885B2")
print("Title:", d.title)
print("Assignees:", d.assignees)
print("CPC:", d.cpc[:4])
print("Dates: priority", d.priority_date, "| filing", d.filing_date, "| publication", d.publication_date)
print("Claims:", len(d.claims), "->", len(d.independent_claims), "independent")
print("Independent claim 1 (head):", d.claims[0].text[:160], "...")
print("Sections:", [s.kind.value for s in d.sections])

Title: Retrieval aware embedding
Assignees: ['Adobe Inc', 'Adobe Inc']
CPC: ['G06N', 'G06N3/00', 'G06N3/02', 'G06N3/08']
Dates: priority 2021-02-10 | filing 2021-02-10 | publication 2024-04-30
Claims: 18 -> 3 independent
Independent claim 1 (head): A method for information retrieval, comprising: generating a dense image embedding for each of a plurality of media objects to be searched; generating a sparse  ...
Sections: ['abstract', 'background', 'summary', 'brief_description_drawings', 'detailed_description', 'other', 'other']


## 5. Provenance & source integrity

Also bundled: genuine **WIPO ST.96 v9.0** example XML (real US Patent 8,936,998) + the official
flattened XSD package (Chapter 02), and a genuine born-digital **patent PDF** (Chapter 03).
Patent text is public record; every artifact records its source.

In [6]:
from pathlib import Path
data = bs.DATA
print("data/xml    :", [p.name for p in sorted((data/'xml').glob('*.xml'))])
print("data/schema :", [p.name for p in sorted((data/'schema').glob('*.zip'))])
print("data/pdf    :", [p.name for p in sorted((data/'pdf').glob('*.pdf'))])
print("\nPROVENANCE.md (first lines):")
print("\n".join(Path(data/'PROVENANCE.md').read_text(encoding='utf-8').splitlines()[:6]))

data/xml    : ['DesignApplication_Example.xml', 'DesignPatentPublication_Example.xml', 'GIApplication_Example.xml', 'PatentPublication_Example.xml', 'ST96_PatentPublication_Example.xml', 'TrademarkApplication_Example.xml']
data/schema : ['ST96_ExampleInstances_V9_0.zip', 'ST96XMLSchema_V9_0_Flattened.zip']
data/pdf    : ['US10083169B1.pdf', 'US9081550B2.pdf']

PROVENANCE.md (first lines):
# Data provenance

All data bundled here is derived from the **public patent record**. Patent documents are
public. Sources and exact retrieval steps are recorded below so the corpus is fully
reproducible via `scripts/build_corpus.py`. Colab/offline runs never re-fetch — they read
these bundled files.


## Chapter invariants

In [7]:
# The corpus loads, is structured, and is non-trivial — the foundation every chapter builds on.
assert manifest["n_docs"] >= 12, "expected >= 12 real patents"
assert manifest["n_claims"] > 100 and manifest["n_sections"] > 50
assert all(d.publication_number and d.claims for d in docs)
assert any(d.independent_claims for d in docs)
assert (bs.DATA / "xml" / "ST96_PatentPublication_Example.xml").exists()
assert list((bs.DATA / "pdf").glob("*.pdf")), "a genuine patent PDF must be bundled"
print("All Chapter 00 invariants hold.")

All Chapter 00 invariants hold.


In [8]:
# === Chapter 00 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['lxml', 'pydantic', 'pandas', 'numpy', 'sentence-transformers', 'faiss-cpu']
print("Chapter 00 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 00 VALIDATION: PASS")

Chapter 00 — environment
  Python : 3.12.10 on Windows 11
  lxml                    : 6.1.1
  pydantic                : 2.13.3
  pandas                  : 3.0.2
  numpy                   : 2.4.2
  sentence-transformers   : 6.0.0
  faiss-cpu               : 1.15.0

CHAPTER 00 VALIDATION: PASS
